In [ ]:
import torch
import os
import random
import numpy as np
import pandas as pd
import os

os.chdir('/Project/100wSampling/')

In [ ]:
def FullyReset(SEED=1):

    os.environ["PYTHONHASHSEED"] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)

    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# Read GeneSets file
GOterms = pd.read_csv('GeneSets.csv')
print(GOterms.head())

GOdict = {}
for i in sorted(set(GOterms['Cluster'])):
    GOdict[i] = np.array(GOterms['GO_term'][GOterms['Cluster'] == i]).tolist()
    print(i, len(GOdict[i]))

In [ ]:
# dataSet: 125989  60542 62321 73168 85258 44408
import pyreadr
gse_id = '62321'
result = pyreadr.read_r(f'./Project/100wSampling/data/GSE{gse_id}_exp.Rdata')
bulk_data = result[f'GSE{gse_id}_exp']
bulk_data = bulk_data.iloc[:,1:].T
bulk_data.columns = result[f'GSE{gse_id}_exp']['symbol']
bulk_data.head()

In [ ]:
result = pyreadr.read_r(f'./Project/100wSampling/data/pData{gse_id}.Rdata')
bulk_label = result[f'pData{gse_id}']
num_samples = len(bulk_label)
num_paires = int(num_samples / 2)
sorted_label =  bulk_label.sort_values(['drop', 'Num'])
sorted_label.index = bulk_label.iloc[:,0]
sorted_label.head()

In [ ]:
# Group samples by phenotype label
g1_label = sorted_label[0:int(num_samples/2)]
g2_label = sorted_label[int(num_samples/2):]

In [ ]:
# Take the intersection of GO terms and expression profile genes
sub_GOdict = {}
for i in sorted(GOdict.keys()):
    overlop_genes = [i for i in set(GOdict[i]).intersection(bulk_data.columns)]
    if len(overlop_genes) >= 30:
        sub_GOdict[i] = sorted(overlop_genes)
        print(f"{i}:\t{len(sub_GOdict[i])}")

In [ ]:
class BatchFuzzyCMeans:
    def __init__(self, n_clusters=2, m=2.0, max_iter=500, tol=1e-6, device="cuda"):
        self.c = n_clusters
        self.m = m
        self.max_iter = max_iter
        self.tol = tol
        self.device = device

    def fit(self, X, y_true):
        if isinstance(X, torch.Tensor):
            X = X.to(self.device, dtype=torch.float32)
        else:
            X = torch.from_numpy(X).to(self.device, dtype=torch.float32)
        batch, n, d = X.shape

        U = torch.rand(batch, n, self.c, device=self.device)
        U = U / U.sum(dim=2, keepdim=True)

        for _ in range(self.max_iter):

            U_old = U.clone()

            um = U ** self.m
            centers = (um.transpose(1, 2) @ X) / um.sum(dim=1, keepdim=True).transpose(1,2)

            dist = torch.cdist(X, centers)
            dist = torch.clamp(dist, min=1e-8)

            power = 2.0 / (self.m - 1)

            inv_dist = dist.pow(-power)
            U = inv_dist / inv_dist.sum(dim=2, keepdim=True)

            if torch.max(torch.abs(U - U_old)) < self.tol:
                break

        self.centers = centers
        self.U_ = U
        self.Class_ = self.U_.argmax(dim=2)
        self.match_label_cluster(y_true)
        self.batch_benchmark(y_true)

    def match_label_cluster(self, y_true):
        n = y_true.size(1)
        pred = self.Class_

        tp = ((y_true == 1) & (pred == 1)).sum(dim=1).float()
        tn = ((y_true == 0) & (pred == 0)).sum(dim=1).float()
        fp = ((y_true == 0) & (pred == 1)).sum(dim=1).float()
        fn = ((y_true == 1) & (pred == 0)).sum(dim=1).float()

        po = (tp + tn) / n
        p_yes_true = (tp + fn) / n
        p_yes_pred = (tp + fp) / n
        p_no_true  = (tn + fp) / n
        p_no_pred  = (tn + fn) / n

        pe = p_yes_true * p_yes_pred + p_no_true * p_no_pred

        kappa = (po - pe) / (1 - pe + 1e-12)

        self.kappa_raw = kappa.clone()
        mask = kappa < 0
        self.U = self.U_.clone()
        self.U[mask] = self.U_[mask].flip(-1)
        self.Class = self.U.argmax(dim=2)
        self.kappa = kappa.abs()

        return self
    
    def batch_benchmark(self, y_true):
        tp = ((y_true == 1) & (self.Class == 1)).sum(dim=1).float()
        tn = ((y_true == 0) & (self.Class == 0)).sum(dim=1).float()
        fp = ((y_true == 0) & (self.Class == 1)).sum(dim=1).float()
        fn = ((y_true == 1) & (self.Class == 0)).sum(dim=1).float()
        
        # Calculate alanced accuracy
        tpr = tp / (tp + fn + 1e-12)
        tnr = tn / (tn + fp + 1e-12)
        self.acc = 0.5 * (tpr + tnr)
        
        # Calculate MCC
        numerator = tp * tn - fp * fn
        denominator = torch.sqrt(
            (tp + fp) * (tp + fn) *
            (tn + fp) * (tn + fn)
        ) + 1e-12
        self.mcc = numerator / denominator

        # Calculate AUC
        B, N = y_true.shape
        y_score = self.U[:, :, 1]
        order = torch.argsort(y_score, dim=1)
        y_sorted = torch.gather(y_true, 1, order)

        n_pos = (y_true == 1).sum(dim=1).float()
        n_neg = (y_true == 0).sum(dim=1).float()

        cum_neg = torch.cumsum((y_sorted == 0).float(), dim=1)

        auc = (cum_neg * (y_sorted == 1).float()).sum(dim=1)

        self.auc = auc / (n_pos * n_neg + 1e-12)
        
        return self 
    

In [ ]:
FullyReset()
for cluster in sorted(sub_GOdict.keys()):

    print(cluster)
    genes_length = len(sub_GOdict[cluster])
    # Probability of drawing each gene, uniformly distributed
    fair_probs = torch.ones(genes_length, device='cuda', dtype=torch.float32) / genes_length

    T_genes = 1000000
    k_genes = 30

    #  Without replacement sampling
    samples1 = torch.multinomial(
        fair_probs.expand(T_genes, -1),
        k_genes,
        replacement=False,
    )
    samples = samples1.cpu().numpy()
    sampled_genes = {
        f"rall_times:{i+1}": [sub_GOdict[cluster][j] for j in row]
        for i, row in enumerate(samples)
    }
    sampledgenes_mat = pd.DataFrame(sampled_genes).T
    print(sampledgenes_mat.head())
    
    T_samples = 36
    k_samples = int(0.7 * num_paires)

    fair_probs = torch.ones(num_paires) / num_paires

    samples2 = torch.multinomial(
        fair_probs.expand(T_samples, -1),
        k_samples,
        replacement=False,
    )

    # Sampling and calculating fuzzy clustering results
    benchmark_res = {}
    data_mat = np.zeros((T_genes, k_samples*2, 30))
    bfcm =  BatchFuzzyCMeans()
    for i, row in enumerate(samples2):
        # Loop for 36 samples
        select_samples = random.sample(g1_label.index[row].tolist()+g2_label.index[row].tolist(), k_samples*2) 
        sub_data = bulk_data.loc[select_samples,:].values                                                      
        sub_label = np.array([0 if j in ['primary', 'PT', 'tumor', 'Primary'] else 1 for j in sorted_label.loc[select_samples, 'drop']]) 
        print(sub_label.mean())
        for _ in range(T_genes):
            data_mat[_] = sub_data[:,samples[_,:]]
        
        newdata = torch.tensor(data_mat, device='cuda', dtype=torch.float32)                                
        newlabel = torch.tensor(sub_label, device='cuda').unsqueeze(0).expand(T_genes, -1)
        
        # Perform fuzzy clustering, align labels, and calculate metrics
        bfcm.fit(newdata, newlabel)
        benchmark_res[f'samples_batch {i}'] = pd.DataFrame({
            f'batch{i}_acc':bfcm.acc.cpu().numpy(),
            f'batch{i}_auc':bfcm.auc.cpu().numpy(),
            f'batch{i}_mcc':bfcm.mcc.cpu().numpy(),
            f'batch{i}_kappa':bfcm.kappa.cpu().numpy(),
        })
        print(f"{(i+1)*100/T_samples:.3f}%")
    
    result = pd.concat(benchmark_res.values(), axis=1 )
    result.index = [f'rall_times:{i+1}' for i in range(T_genes)]
    print(result.head())

    result.to_csv(f'./{cluster}/GSE{gse_id}/Benchmark_100w.csv')
    sampledgenes_mat.to_csv(f'./{cluster}/GSE{gse_id}/GenesSampledRes_100w.csv')